# Feature Engineering — Dầu & Cửa hàng

| Nhóm | Feature | Mô tả |
|---|---|---|
| Oil | `oil_price` | Giá dầu WTI đã ffill |
| Oil | `oil_price_lag_7/14` | Lag 7/14 ngày |
| Oil | `oil_price_rolling_mean_28` | Rolling mean 28 ngày (shift-1 safe) |
| Oil | `oil_price_change_pct` | % thay đổi vs tuần trước |
| Store | `store_type_enc` | Label encode loại store (A–E → 0–4) |
| Store | `city_freq`, `state_freq` | Frequency encode vị trí |

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [ ]:
import sys
from pathlib import Path

def _find_root():
    """Tìm project root chứa config.py."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'config.py').exists():
            return p
    raise RuntimeError("Không tìm thấy project root. Mở Jupyter từ thư mục gốc của project.")

_root = _find_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import PROCESSED_DIR

df_train  = pd.read_csv(PROCESSED_DIR / 'train_cleaned.csv')
df_test   = pd.read_csv(PROCESSED_DIR / 'test_cleaned.csv')
df_oil    = pd.read_csv(PROCESSED_DIR / 'cleaned_oil.csv')
df_stores = pd.read_csv(PROCESSED_DIR / 'stores_cleaned.csv')

print(f'Train shape  : {df_train.shape}')
print(f'Test shape   : {df_test.shape}')
print(f'Oil shape    : {df_oil.shape}')
print(f'Stores shape : {df_stores.shape}')

---
## 1. Oil Features

In [3]:
def create_oil_features(oil_df):
    oil = oil_df.copy().rename(columns={"dcoilwtico":"oil_price"})
    oil["date"] = pd.to_datetime(oil["date"])
    oil = oil[["date","oil_price"]].sort_values("date").reset_index(drop=True)

    full_date_range = pd.DataFrame({"date": pd.date_range(start=oil["date"].min(), end=oil["date"].max(), freq="D")})
    oil = full_date_range.merge(oil, on="date", how="left")

    oil["oil_price"] = oil["oil_price"].ffill()
    oil["oil_price"] = oil["oil_price"].bfill()

    oil["oil_price_lag_7"]  = oil["oil_price"].shift(7)
    oil["oil_price_lag_14"] = oil["oil_price"].shift(14)
    oil["oil_price_rolling_mean_28"] = oil["oil_price"].shift(1).rolling(28, min_periods=7).mean()
    oil["oil_price_change_pct"] = oil["oil_price"].pct_change(periods=7)

    return oil

# Test the function
oil_featured = create_oil_features(df_oil)
print(f'Oil featured shape: {oil_featured.shape}')
print(f'Columns: {oil_featured.columns.tolist()}')
oil_featured.head(10)

Oil featured shape: (1704, 6)
Columns: ['date', 'oil_price', 'oil_price_lag_7', 'oil_price_lag_14', 'oil_price_rolling_mean_28', 'oil_price_change_pct']


,date,oil_price,oil_price_lag_7,oil_price_lag_14,oil_price_rolling_mean_28,oil_price_change_pct
0,2013-01-01,93.14,NaN,NaN,NaN,NaN
1,2013-01-02,93.14,NaN,NaN,NaN,NaN
2,2013-01-03,92.97,NaN,NaN,NaN,NaN
3,2013-01-04,93.12,NaN,NaN,NaN,NaN
4,2013-01-05,93.12,NaN,NaN,NaN,NaN
5,2013-01-06,93.12,NaN,NaN,NaN,NaN
6,2013-01-07,93.20,NaN,NaN,NaN,NaN
7,2013-01-08,93.21,93.14,NaN,93.115714,0.000752
8,2013-01-09,93.08,93.14,NaN,93.127500,-0.000644
9,2013-01-10,93.81,92.97,NaN,93.122222,0.009035


---
## 2. Store Encoding

In [4]:
def encode_store_features(train, test):

    train, test = train.copy(), test.copy()

    # 1. Label encode store_type (fit on train only)
    le = LabelEncoder()
    train["store_type_enc"] = le.fit_transform(train["type"])
    test["store_type_enc"] = le.transform(test["type"])

    # 2. Frequency encode city/state
    for col in ["city", "state"]:
        freq = train[col].value_counts(normalize=True)
        train[f"{col}_freq"] = train[col].map(freq)
        test[f"{col}_freq"] = test[col].map(freq).fillna(0)

    # 3. cluster keep the same

    return train, test


# Merge store info into train/test before encoding
store_cols = ['store_nbr', 'type', 'city', 'state']
train_with_store = df_train.merge(df_stores[store_cols], on='store_nbr', how='left')
test_with_store  = df_test.merge(df_stores[store_cols], on='store_nbr', how='left')

train_encoded, test_encoded = encode_store_features(train_with_store, test_with_store)

print(f'Train encoded shape: {train_encoded.shape}')
print(f'Test encoded shape : {test_encoded.shape}')
print(f'\nNew columns: store_type_enc, city_freq, state_freq')
train_encoded[['store_nbr', 'type', 'store_type_enc', 'city_freq', 'state_freq']].drop_duplicates('store_nbr').head(10)

Train encoded shape: (3000888, 12)
Test encoded shape : (28512, 11)

New columns: store_type_enc, city_freq, state_freq


,store_nbr,type,store_type_enc,city_freq,state_freq
0,1,D,3,0.333333,0.351852
33,10,C,2,0.333333,0.351852
66,11,B,1,0.018519,0.351852
99,12,C,2,0.037037,0.037037
132,13,C,2,0.037037,0.037037
165,14,C,2,0.018519,0.018519
198,15,C,2,0.018519,0.018519
231,16,C,2,0.055556,0.055556
264,17,C,2,0.333333,0.351852
297,18,B,1,0.333333,0.351852


---
## 3. Transaction Features (TODO)

---
## 4. Feature Name Registry

In [5]:
OIL_FEATURE_NAMES = [
    'oil_price',
    'oil_price_lag_7',
    'oil_price_lag_14',
    'oil_price_rolling_mean_28',
    'oil_price_change_pct',
]

STORE_FEATURE_NAMES = [
    'store_type_enc',
    'city_freq',
    'state_freq',
]

# TODO: Uncomment khi transaction features được triển khai
# TRANSACTION_FEATURE_NAMES = [
#     'transactions_lag_7',
#     'transactions_rolling_mean_7',
# ]

ALL_OIL_STORE_FEATURES = OIL_FEATURE_NAMES + STORE_FEATURE_NAMES

print(f'Oil features   : {len(OIL_FEATURE_NAMES)}')
print(f'Store features : {len(STORE_FEATURE_NAMES)}')
print(f'Total features : {len(ALL_OIL_STORE_FEATURES)}')
print(f'\nAll features: {ALL_OIL_STORE_FEATURES}')

Oil features   : 5
Store features : 3
Total features : 8

All features: ['oil_price', 'oil_price_lag_7', 'oil_price_lag_14', 'oil_price_rolling_mean_28', 'oil_price_change_pct', 'store_type_enc', 'city_freq', 'state_freq']


---
## 5. Chạy Pipeline

In [6]:
# --- Step 1: Oil features ---
oil_featured = create_oil_features(df_oil)

# Merge oil features vào train
df_train['date'] = pd.to_datetime(df_train['date'])
df_result = df_train.merge(oil_featured[['date'] + OIL_FEATURE_NAMES], on='date', how='left')

print(f'After oil merge shape: {df_result.shape}')
print(f'Oil NaN counts:')
print(df_result[OIL_FEATURE_NAMES].isna().sum())

# --- Step 2: Store encoding ---
store_cols = ['store_nbr', 'type', 'city', 'state']
df_result = df_result.merge(df_stores[store_cols], on='store_nbr', how='left')

df_test_tmp = df_test.copy()
df_test_tmp = df_test_tmp.merge(df_stores[store_cols], on='store_nbr', how='left')

df_result, df_test_out = encode_store_features(df_result, df_test_tmp)

print(f'\nAfter store encoding shape: {df_result.shape}')
print(f'\n=== Feature Summary ===')
print(df_result[ALL_OIL_STORE_FEATURES].describe().T[['mean', 'min', 'max']])
print(f'\n=== Sample rows ===')
df_result[['date', 'store_nbr'] + ALL_OIL_STORE_FEATURES].head(10)

After oil merge shape: (3000888, 11)
Oil NaN counts:
oil_price                        0
oil_price_lag_7              12474
oil_price_lag_14             24948
oil_price_rolling_mean_28    12474
oil_price_change_pct         12474
dtype: int64

After store encoding shape: (3000888, 17)

=== Feature Summary ===
                                mean        min         max
oil_price                  67.924899  26.190000  110.620000
oil_price_lag_7            68.009296  26.190000  110.620000
oil_price_lag_14           68.083719  26.190000  110.620000
oil_price_rolling_mean_28  68.202760  30.210000  108.076429
oil_price_change_pct       -0.001698  -0.171989    0.287284
store_type_enc              2.000000   0.000000    4.000000
city_freq                   0.149520   0.018519    0.333333
state_freq                  0.182442   0.018519    0.351852

=== Sample rows ===


,date,store_nbr,oil_price,oil_price_lag_7,oil_price_lag_14,oil_price_rolling_mean_28,oil_price_change_pct,store_type_enc,city_freq,state_freq
0,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
1,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
2,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
3,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
4,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
5,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
6,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
7,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
8,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852
9,2013-01-01,1,93.14,NaN,NaN,NaN,NaN,3,0.333333,0.351852


In [ ]:
# CSV write disabled — tất cả features được merge trong build_train_final.py.
# Để tạo CSV cho model training:
#   python scripts/build_train_final.py   →  data/processed/train_final.csv
#   python scripts/create_splits.py       →  data/processed/splits/
print('[SKIPPED] CSV write disabled')
print(f'Feature dataframe shape : {df_result.shape}')